# Chapter 7. BERT 첫 만남 — `pipeline` 한 줄과 그 안의 4단계

**목표**: sklearn 시대를 마치고 `transformers` 라이브러리를 만납니다. **5줄짜리 코드** 로 사전학습된 DistilBERT를 돌려보고, 그 한 줄 뒤에 어떤 일이 일어났는지 단계별로 풀어 헤칩니다.

**환경**: Google Colab — **T4 GPU 권장** (런타임 → 런타임 유형 변경 → T4 GPU). 이번 챕터부터 GPU 메모리(VRAM) 추적이 등장하니 GPU 런타임에서 돌리면 모델 로드 → VRAM 증가가 한눈에 보입니다. CPU 런타임도 추론 자체는 동작하지만 `!nvidia-smi` 셀은 에러납니다.

**예상 소요 시간**: 약 10분 (학습 없음, 추론만)

---

## 학습 흐름

1. 🚀 **실습**: `pipeline("sentiment-analysis")` 한 줄로 감성 분석 돌리기
2. 🔬 **해부**: `pipeline` 안에서 일어나는 3단계 (tokenizer / model / post-processing)
3. 🛠️ **변형**: `pipeline` 없이 같은 일을 4단계로 직접 재현

## 📊 변화추적표

**Phase 1 시작** — sklearn 시대 끝, `transformers` 등장.

| Ch | 모델 | 토크나이저 | 데이터 | Output Head | Activation | Loss |
|---|---|---|---|---|---|---|
| 1 | (TF-IDF) | `TfidfVectorizer()` | Yelp 5,000 | — | — | — |
| 2 | `LinearRegression()` | `TfidfVectorizer()` | Yelp (별점 1-5) | (1차원) | 없음 | `MSELoss` |
| 3 | `LogisticRegression()` | `TfidfVectorizer()` | Yelp 이진화 | (1차원) | sigmoid | `BCEWithLogitsLoss` |
| 4 | `LogisticRegression()` (multinomial 자동) | `TfidfVectorizer()` | Yelp 이진화 | (2차원) | softmax | `CrossEntropyLoss` |
| 5 | `LogisticRegression()` (multinomial 자동) | `TfidfVectorizer()` | Yelp 5클래스 | (5차원) | softmax | `CrossEntropyLoss` |
| 6 | `OneVsRestClassifier(LogisticRegression())` | `TfidfVectorizer()` | Yelp + 측면 합성 | (5차원) | per-label sigmoid | per-label `BCEWithLogitsLoss` |
| **7 ← 여기** | `pipeline("sentiment-analysis")` | `AutoTokenizer.from_pretrained(...)` | 간단 영어 예시 | **사전학습 헤드** | softmax | — (추론만) |

전체 20챕터 표는 [루트 README.md](https://github.com/yoon-gu/neuqes-101#챕터별-변화추적표)를 참고하세요.

## 🔄 변경점 (Diff from Ch 6)

| 축 | Ch 6 | Ch 7 |
|---|---|---|
| 라이브러리 | `sklearn` | **`transformers`** (Hugging Face) |
| 모델 | `OneVsRestClassifier(LogisticRegression())` (학습) | **`pipeline("sentiment-analysis")`** (사전학습 + 추론) |
| 토크나이저 | `TfidfVectorizer()` (단어 단위 어휘 학습) | **`AutoTokenizer`** (사전학습된 WordPiece) |
| 학습 단계 | sklearn `fit()` 한 번에 학습 | **학습 없음** — 사전학습 가중치 로드 후 추론만 |
| 데이터 | Yelp 5,000건 | 간단 영어 예시 문장 (분해 시연용) |
| 하드웨어 | CPU | CPU 또는 T4 GPU (이번 챕터는 추론만이라 어느 쪽도 OK) |

**왜 학습 없이 시작하나?** Phase 1 첫 챕터는 `transformers` 의 *추상화 계층* 을 익히는 데 집중합니다. `pipeline` 한 줄 뒤에 토크나이저·모델·후처리 3단계가 어떻게 굴러가는지 손에 잡히면, Ch 8(Tokenizer/Datasets 해부)와 Ch 9(BERT 회귀 첫 학습)에서 `Trainer` 가 등장할 때 코드를 *읽을* 수 있습니다.

## 🔤 토크나이저 노트 — 첫 WordPiece 등장

이번 챕터의 토크나이저는 **사전학습된 WordPiece**. Phase 0의 `TfidfVectorizer` 와 *완전히 다른 패러다임* 입니다.

| 비교 | TF-IDF (Phase 0) | WordPiece (Phase 1+) |
|---|---|---|
| 분리 단위 | 단어 (whitespace + 정규식) | **서브워드** (자주 등장하는 문자 시퀀스) |
| 어휘 출처 | 학습 데이터에서 그때그때 학습 | **사전학습된 30,522개 어휘** (BERT 학습 시 정해짐) |
| OOV 처리 | 그냥 무시 | `[UNK]` 또는 더 작은 서브워드로 분해 |
| 특수 토큰 | 없음 | `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]` 등 |
| 출력 | sparse vector (V차원, 거의 0) | 정수 ID 시퀀스 + attention mask |

같은 문장 `"I love using Hugging Face!"` 가 어떻게 토큰화되는지 곧 직접 확인합니다 (Step 2). `##` 접두사가 보이는 단어는 어디고, 왜 그렇게 쪼개졌는지도 같이 봅니다.

> **다음 챕터(Ch 8)**: 같은 WordPiece 토크나이저를 *깊게* — `padding`, `truncation`, `max_length` 옵션과 `datasets` 라이브러리 메모리 효율까지.

## 0. 환경 준비

Colab에는 `transformers`가 보통 설치돼 있지만, 최신 버전을 보장하기 위해 한 번 설치합니다.

In [ ]:
!pip install -q transformers

In [ ]:
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (inference works in this chapter; skip the nvidia-smi cells)")

### `!nvidia-smi` — GPU 메모리(VRAM) 실시간 추적

이번 챕터부터 학습·추론 코드가 GPU에 모델을 올리기 시작합니다. **`!nvidia-smi`** 는 NVIDIA에서 제공하는 명령행 도구로, 현재 GPU의 VRAM 사용량·온도·전력을 한 번에 보여줍니다. Colab 셀에서 `!` 접두사로 호출 가능.

T4의 총 VRAM은 **약 15.36 GB** (= 15,360 MiB). 모델·옵티마이저·activation을 모두 이 안에 담아야 합니다 — Ch 9 이후 학습 chapter에서는 이 한도와 자주 부딪히게 되어요.

**baseline** — 아직 아무 모델도 안 올린 상태:

In [ ]:
!nvidia-smi

**무엇을 봐야 하나** — 출력 가운데 줄 `Memory-Usage` 칸:

```
| ... |  XXX MiB / 15360MiB | ...
        └─ used    └─ total
```

- 처음엔 ~3-200 MiB 정도. CUDA 컨텍스트가 잡혀 있는 만큼만.
- 모델을 GPU에 올릴 때마다 `used` 가 증가합니다.
- `Volatile GPU-Util` 은 *현재* GPU가 일하는 비율 — 학습 중에는 90~100% 가까이.

**Python으로도 확인 가능** (셀 내부에서 변수로 받고 싶을 때):

```python
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**2
    total = torch.cuda.get_device_properties(0).total_memory / 1024**2
    print(f"GPU memory: {used:.0f} / {total:.0f} MiB")
```

> Tip: `!nvidia-smi` 는 *시스템 전체* VRAM을 보여주고, `torch.cuda.memory_allocated()` 는 *현재 PyTorch 프로세스* 의 할당량만 보여줍니다 — 후자는 캐시·예약 메모리는 빼고 실제 텐서가 점유한 양에 가깝습니다.

## 1. 🚀 실습: 일단 돌려봅시다

Hugging Face의 `pipeline` 은 **"모델 다운로드 → 토큰화 → 추론 → 결과 후처리"** 를 한 줄로 묶어주는 함수입니다.

감성 분석(sentiment analysis)부터 시작합니다. **GPU가 있으면 `device=0` 으로 명시** — 그래야 모델이 VRAM에 올라가서 nvidia-smi 변화가 보입니다 (기본은 CPU).

In [ ]:
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1   # 0 = GPU index, -1 = CPU
classifier = pipeline("sentiment-analysis", device=DEVICE)
classifier("I love using Hugging Face! It's so simple.")

**DistilBERT(SST-2)가 VRAM에 올라간 직후의 nvidia-smi:**

In [ ]:
!nvidia-smi

baseline과 비교하면 `Memory-Usage` 가 늘어났을 겁니다. **얼마나 늘어났을까?** — 모델 파라미터 수에서 거꾸로 추정해봅니다.

In [ ]:
def model_size_summary(model, dtype_bytes=4):
    # 파라미터 개수와 dtype 기준 예상 메모리 반환. fp32 = 4 bytes, fp16/bf16 = 2 bytes.
    n_params = sum(p.numel() for p in model.parameters())
    size_mb = n_params * dtype_bytes / 1024**2
    return n_params, size_mb

n, size_mb = model_size_summary(classifier.model)
print(f"DistilBERT (SST-2) parameters:")
print(f"  count:           {n:>13,}  ({n/1e6:.1f} M)")
print(f"  fp32 size:       {size_mb:>10.1f} MB  (= params × 4 bytes)")

**파라미터 수와 VRAM의 관계**

가중치 한 개는 *한 개의 부동소수* — fp32면 4 bytes, fp16/bf16이면 2 bytes를 차지합니다. 그래서:

$$\text{모델 가중치 크기} \approx \text{파라미터 수} \times \text{dtype bytes}$$

| dtype | bytes/param | DistilBERT(67M) 예상 크기 |
|---|---|---|
| **fp32** (기본 학습) | 4 | ~255 MB |
| **fp16 / bf16** (mixed precision) | 2 | ~128 MB |
| **int8** (양자화 추론) | 1 | ~64 MB |

**그런데 nvidia-smi는 파라미터 크기보다 *더 많이* 늘어나는데요?** 차이의 정체:

- **PyTorch CUDA 컨텍스트** (~150-300 MB): 라이브러리·드라이버가 GPU 점유 시 한 번 잡는 고정 비용. 첫 모델 로드에서만 보이고 이후엔 누적되지 않음.
- **CUDA 캐시 할당자** (수십 MB): PyTorch가 자주 쓰일 텐서를 캐싱.
- **추론 중 일시 activation**: forward pass에서 layer 사이 중간 결과. 추론은 작지만 학습은 큼.

**학습이 되면 메모리는 더 커집니다** — Adam 옵티마이저는 모델당 *추가로 2배* (1차·2차 모멘텀)를 더 들고, gradient도 *모델 크기만큼* 한 벌 — 즉 학습 중엔 **fp32 기준 파라미터 × 4배 정도** 의 VRAM이 필요합니다. Ch 9에서 다시 다룹니다.

**참고**: 처음 실행 시 모델 다운로드(약 250MB)에 30초~1분 정도 걸립니다. 두 번째부터는 캐시되어 즉시 실행.

여러 문장도 한 번에:

In [ ]:
results = classifier([
    "This movie was fantastic.",
    "Worst experience ever.",
    "It was okay, nothing special.",
])
for r in results:
    print(r)

### 다른 task도 같은 패턴

`pipeline` 의 첫 인자만 바꾸면 다른 NLP 작업을 즉시 할 수 있습니다.

In [ ]:
# 텍스트 생성 (GPT-2)
generator = pipeline("text-generation", model="gpt2", device=DEVICE)
generator("Hugging Face is", max_length=30, num_return_sequences=1)

In [ ]:
# 마스크 채우기 (BERT)
unmasker = pipeline("fill-mask", model="bert-base-uncased", device=DEVICE)
unmasker("Hugging Face is a [MASK] for NLP.")

**3개 pipeline(DistilBERT + GPT-2 + BERT-base)이 모두 VRAM에 쌓인 상태:**

In [ ]:
!nvidia-smi

**파라미터 수 합계로 예측한 모델 가중치 vs 실제 VRAM 증가** — 차이가 PyTorch 오버헤드입니다.

In [ ]:
models_info = {
    "DistilBERT (SST-2)":       classifier.model,
    "GPT-2":                    generator.model,
    "BERT-base-uncased":        unmasker.model,
}

print(f"{'model':>30}  {'params':>12}  {'fp32 size':>14}")
print("-" * 65)
total_params, total_size = 0, 0.0
for name, m in models_info.items():
    n, sz = model_size_summary(m)
    total_params += n
    total_size += sz
    print(f"{name:>30}  {n/1e6:>9.1f} M  {sz:>11.1f} MB")
print("-" * 65)
print(f"{'total':>30}  {total_params/1e6:>9.1f} M  {total_size:>11.1f} MB")
print(f"{'in GB':>30}  {' '*12}  {total_size/1024:>11.2f} GB")

**실제 nvidia-smi VRAM 사용량과 비교** — 위 합계(~수백 MB)에 **PyTorch CUDA 컨텍스트 + 캐시 할당자 ~ 200-400 MB** 를 더한 값이 nvidia-smi 의 `Memory-Usage` 와 비슷할 겁니다.

> 모델별 파라미터 차이가 흥미롭습니다.
> - **DistilBERT (~67M)**: BERT-base에서 layer를 절반으로 줄여 학습한 경량화 모델. 추론 속도 ~2배.
> - **GPT-2 small (~124M)**: 파라미터는 BERT-base보다 약간 많고, 어휘(50K)도 더 큼.
> - **BERT-base (~110M)**: BERT 표준 사이즈.

**메모리를 비우는 표준 패턴** — 더 이상 안 쓰는 모델은:

```python
import gc, torch
del generator, unmasker         # 파이썬 참조 제거 (refcount=0)
gc.collect()                    # 가비지 컬렉션
if torch.cuda.is_available():
    torch.cuda.empty_cache()    # CUDA 캐시 비우기 (예약된 캐시 반환)
```

T4 메모리(15.36 GB)는 작은 추론 모델 여러 개를 무리 없이 담지만, BERT-large(~340M, fp32 ~1.4 GB)나 학습 시 *옵티마이저 + gradient* 까지 올리면 빠르게 한도에 도달하니 항상 nvidia-smi로 잔여 VRAM 확인하는 습관이 좋습니다.

## 2. 🔬 해부: pipeline 안에서는 뭐가 일어났을까?

`pipeline("sentiment-analysis")` 한 줄이 사실은 **3단계** 로 구성됩니다.

```
입력 텍스트
   ↓ [1] Tokenizer  (텍스트 → 숫자 ID)
input_ids, attention_mask
   ↓ [2] Model      (숫자 → 로짓)
logits
   ↓ [3] Post-processing (로짓 → 라벨)
{'label': 'POSITIVE', 'score': 0.9998}
```

### 등장 인물 정리

| 컴포넌트 | 라이브러리 | 역할 |
|---|---|---|
| `pipeline` | `transformers` | 위 3단계를 묶은 wrapper |
| Tokenizer | `transformers` (내부적으로 `tokenizers`) | 텍스트를 모델이 먹을 수 있는 숫자로 변환 |
| Model | `transformers` + `torch` | 실제 신경망 forward 연산 |

현재 `classifier` 객체가 어떤 모델/토크나이저를 사용하는지 확인합니다.

In [ ]:
print(f"Model:               {classifier.model.config._name_or_path}")
print(f"Model class:         {type(classifier.model).__name__}")
print(f"Tokenizer class:     {type(classifier.tokenizer).__name__}")
print(f"Label mapping:       {classifier.model.config.id2label}")

기본 모델은 **`distilbert-base-uncased-finetuned-sst-2-english`** 입니다.

- **DistilBERT**: BERT를 40% 작게 만든 경량화 모델 (학생 모델, 지식 증류로 만듦).
- **SST-2**: 영화 리뷰 감성 분류 데이터셋 (Stanford Sentiment Treebank).
- 즉, "BERT를 영화 리뷰 데이터로 파인튜닝한 모델"입니다 — 우리가 Ch 9에서 직접 할 작업의 *완성된* 버전.

## 3. 🛠️ 변형: pipeline 없이 직접 해보기

이제 `pipeline` 이 감춰뒀던 단계를 **직접 한 줄씩 실행** 합니다. 이 부분을 이해하면 앞으로 모든 모델을 자유롭게 다룰 수 있습니다.

### Step 1: Tokenizer와 Model 직접 로드

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# GPU가 있으면 모델을 VRAM으로 이동 (직접 로드는 default가 CPU라 명시 필요)
if torch.cuda.is_available():
    model = model.to("cuda")

print("Loaded")
print(f"  tokenizer class: {type(tokenizer).__name__}")
print(f"  model class:     {type(model).__name__}")
print(f"  model device:    {next(model.parameters()).device}")

**`pipeline` 위에 추가로 *같은 DistilBERT* 가 올라간 상태** — VRAM이 또 한 번 늘어났습니다. 같은 가중치라도 별도 객체이면 별도 메모리.

In [ ]:
!nvidia-smi

> **잠깐, `Auto`가 뭔가요?**
>
> `AutoTokenizer`, `AutoModel...` 같은 `Auto` 계열 클래스는 모델 이름만 주면 **알아서 적합한 클래스를 골라주는 팩토리** 입니다.
>
> - DistilBERT 모델 → `DistilBertTokenizer`, `DistilBertForSequenceClassification`
> - BERT 모델 → `BertTokenizer`, `BertForSequenceClassification`
> - GPT-2 모델 → `GPT2Tokenizer`, `GPT2LMHeadModel`
>
> 직접 `BertTokenizer.from_pretrained(...)`라고 써도 되지만, `AutoTokenizer` 를 쓰면 모델만 바꿔도 코드가 그대로 동작합니다.

### Step 2: 텍스트 → 숫자 (Tokenization)

In [ ]:
text = "I love using Hugging Face!"

# 토큰화 결과 살펴보기
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")

# 모델 입력용 텐서 만들기
inputs = tokenizer(text, return_tensors="pt")
print("\nModel inputs:")
for key, value in inputs.items():
    print(f"  {key}: {value}")

**관찰 포인트**
- `input_ids`: 각 토큰의 정수 ID. 맨 앞 `101`은 `[CLS]`, 맨 뒤 `102`는 `[SEP]` 특수 토큰.
- `attention_mask`: `1`이면 "이 위치는 진짜 토큰", `0`이면 "패딩이니 무시하라"는 신호 (이번엔 패딩 없음 → 모두 1).
- `tokenize()` 와 `tokenizer()` 의 차이 — 전자는 토큰 문자열만 반환, 후자는 모델 입력용 텐서까지 다 만들어줌.

In [ ]:
# input_ids를 다시 토큰으로 디코딩해서 확인
print(f"{'ID':>5}    token")
print("-" * 30)
for token_id in inputs["input_ids"][0]:
    token = tokenizer.decode([token_id])
    print(f"{token_id.item():>5}    {token!r}")

### Step 3: 숫자 → 로짓 (Model forward)

In [ ]:
# 입력 텐서도 모델과 같은 device로 이동시켜야 함 (CPU↔GPU 혼합 forward는 에러)
inputs_on_device = {k: v.to(model.device) for k, v in inputs.items()}

# 추론할 때는 gradient 계산을 끄는 것이 메모리/속도에 좋음
with torch.no_grad():
    outputs = model(**inputs_on_device)

print(f"Output object: {type(outputs).__name__}")
print(f"Logits shape:  {outputs.logits.shape}  (batch=1, classes=2: NEGATIVE, POSITIVE)")
print(f"Logits:        {outputs.logits}")
print(f"Logits device: {outputs.logits.device}")

로짓(logits)은 모델이 뱉은 **정규화되지 않은 점수**. shape `[1, 2]`는 "배치 1개, 클래스 2개"를 의미합니다.

여기서 잠깐 — 익숙하지 않나요? Ch 4의 *softmax + 2차원 head* 구조와 정확히 같습니다. BERT는 사전학습된 *심층* 모델일 뿐, 마지막 분류 헤드는 sklearn에서 본 것과 본질이 같습니다.

### Step 4: 로짓 → 확률/라벨 (Post-processing)

In [ ]:
# softmax로 확률 변환
probs = torch.softmax(outputs.logits, dim=-1)
print(f"Probabilities: {probs}")

# 가장 높은 확률의 클래스 인덱스
predicted_class_id = probs.argmax(dim=-1).item()
predicted_label = model.config.id2label[predicted_class_id]
predicted_score = probs[0, predicted_class_id].item()

print(f"\nFinal result: {{'label': '{predicted_label}', 'score': {predicted_score:.4f}}}")

**잠깐 — `transformers` 안에는 softmax 함수가 없나요?**

없습니다. Hugging Face는 *모델·토크나이저·학습 루프* 를 제공하고, **수치 연산은 PyTorch에 위임**합니다 (혹은 TensorFlow / JAX). 그래서 후처리(softmax, argmax, log 등)는 `torch.*` 를 직접 부릅니다.

PyTorch 안에는 같은 softmax를 표현하는 세 가지 형태가 있습니다 — 결과는 모두 동일하고 *어디서 호출하느냐* 만 다릅니다.

In [ ]:
import torch.nn.functional as F

# 형태 1: torch.softmax   ← 이번 챕터에서 쓴 형태 (텐서 메서드 스타일)
p1 = torch.softmax(outputs.logits, dim=-1)

# 형태 2: F.softmax        ← functional 네임스페이스 (가장 흔하게 보이는 PyTorch 패턴)
p2 = F.softmax(outputs.logits, dim=-1)

# 형태 3: nn.Softmax       ← 모듈 형태 (모델 내부에 layer로 박을 때 사용)
softmax_module = torch.nn.Softmax(dim=-1)
p3 = softmax_module(outputs.logits)

print(f"Do all three forms agree?")
print(f"  torch.softmax vs F.softmax:    {torch.allclose(p1, p2)}")
print(f"  torch.softmax vs nn.Softmax:   {torch.allclose(p1, p3)}")
print(f"\n  values: {p1}")

**보너스 — 학습 코드에서 자주 보이는 `log_softmax`**

수치적 안정성 때문에 학습 시에는 `softmax → log` 두 단계 대신 **`log_softmax` 한 번** 으로 묶는 게 표준입니다 (`CrossEntropyLoss` 가 내부에서 이렇게 함).

```python
# softmax 후 log를 따로 (수치 불안정 가능)
log_probs_unstable = torch.log(torch.softmax(logits, dim=-1))

# log_softmax 한 번 (안정적)
log_probs_stable = F.log_softmax(logits, dim=-1)
```

추론 시에는 그냥 `softmax` 가 깔끔합니다 — 확률이 직접 필요하니까요. **학습 시** `CrossEntropyLoss(logits, target)` 는 내부적으로 logit에 `log_softmax` 를 적용하고 NLL을 더하므로, *softmax를 직접 부를 일이 없습니다* (Ch 9 이후 자주 등장).

In [ ]:
# pipeline이 한 줄로 해주던 일을 4단계로 직접 재현했습니다. 결과를 비교해봅시다.
print(f"pipeline result:    {classifier(text)}")
print(f"manual 4-step:      [{{'label': '{predicted_label}', 'score': {predicted_score:.4f}}}]")

## 🔍 보너스: 토크나이저마다 어휘가 다르다

지금까지는 DistilBERT의 WordPiece 토크나이저 *하나* 만 봤습니다. 그런데 모델이 바뀌면 토크나이저도 바뀌고, **같은 문장이 완전히 다른 토큰 리스트로 쪼개집니다** — 어휘 사전이 사전학습 단계에서 따로 만들어졌기 때문이에요.

세 가지 대표 토크나이저를 나란히 비교합니다.

| 모델 | 알고리즘 | 어휘 크기 | 대소문자 |
|---|---|---|---|
| `distilbert-base-uncased` | **WordPiece** | 30,522 | 모두 소문자로 |
| `bert-base-cased` | **WordPiece** | 28,996 | 대소문자 유지 |
| `gpt2` | **BPE** (Byte Pair Encoding) | 50,257 | 대소문자 유지 |

WordPiece와 BPE는 둘 다 *서브워드 알고리즘* 이지만 학습·표기 방식이 달라서 토큰 모양이 시각적으로도 구분됩니다 — `##` 접두사 vs `Ġ` (공백) 접두사.

In [ ]:
# 토크나이저 3종 로드 (모델 가중치는 안 받고 토크나이저 파일만 ~수백 KB)
tokenizer_specs = {
    "distilbert-base-uncased": AutoTokenizer.from_pretrained("distilbert-base-uncased"),
    "bert-base-cased":         AutoTokenizer.from_pretrained("bert-base-cased"),
    "gpt2":                    AutoTokenizer.from_pretrained("gpt2"),
}

print(f"{'model':>28}  {'vocab_size':>10}  {'class':>32}")
print("-" * 76)
for name, tok in tokenizer_specs.items():
    print(f"{name:>28}  {tok.vocab_size:>10,}  {type(tok).__name__:>32}")

In [ ]:
sample_sentences = [
    "I love using Hugging Face!",
    "Tokenization is fascinating.",
]

for sent in sample_sentences:
    print(f"Input: {sent!r}")
    for name, tok in tokenizer_specs.items():
        tokens = tok.tokenize(sent)
        print(f"  {name:>28}  ({len(tokens)} tokens) {tokens}")
    print()

### 특수 토큰(special token)이란

`[CLS]`, `[SEP]` 같은 토큰은 *문장 텍스트* 가 아니라 **모델에 신호를 주기 위해 사전학습 단계에서 정해진 약속** 입니다. 어휘 사전에 별도 ID로 들어 있고, 토크나이저가 입력에 자동으로 붙입니다.

| 토큰 | 풀이름 | 위치 | 역할 |
|---|---|---|---|
| `[CLS]` | Classification | 모든 입력 *맨 앞* | 분류 헤드는 *이 위치* 의 hidden state를 사용. attention을 통해 전체 문장 정보가 [CLS]로 모이도록 학습됨. |
| `[SEP]` | Separator | 문장 끝, 두 문장 사이 | 한 문장 입력엔 `[CLS] ... [SEP]`. 두 문장이면 `[CLS] A [SEP] B [SEP]` (NSP·QA·NLI 등). |
| `[PAD]` | Padding | 짧은 문장 끝 | 배치 안 문장 길이를 맞추는 더미 토큰. **`attention_mask=0`** 으로 표시해 모델이 무시. |
| `[UNK]` | Unknown | 어디든 | 어휘 사전에 없는 토큰. WordPiece는 거의 항상 더 작은 서브워드로 쪼개므로 실제 출현은 드뭄. |
| `[MASK]` | Mask | 사전학습 시 입력 일부 | BERT 사전학습의 *Masked LM* — 입력 토큰 15%를 `[MASK]` 로 가리고 모델이 맞추도록. 추론 시엔 거의 안 등장(fill-mask 데모 제외). |

**autoregressive 모델 (GPT-2)** 은 `[CLS]/[SEP]` 가 없습니다 — 다음 토큰을 *순서대로* 예측하는 구조라 문장 시작/끝 마커가 별도로 필요 없고, `<|endoftext|>` 라는 단일 토큰이 BOS/EOS 역할을 겸합니다.

이 약속은 *모델별로 다릅니다*. RoBERTa는 `<s>`, `</s>` 를, T5는 `<pad>`, `<extra_id_0>` 등을 씁니다 — `tokenizer.special_tokens_map` 으로 한 번에 확인 가능.

In [ ]:
# 특수 토큰: 모델마다 어떤 token을 [CLS]/[SEP]/[PAD]/[UNK] 자리에 두는지
print(f"{'model':>28}  {'BOS/CLS':>16}  {'EOS/SEP':>16}  {'PAD':>10}  {'UNK':>10}")
print("-" * 90)
for name, tok in tokenizer_specs.items():
    cls = tok.cls_token if tok.cls_token else (tok.bos_token or "—")
    sep = tok.sep_token if tok.sep_token else (tok.eos_token or "—")
    pad = tok.pad_token or "—"
    unk = tok.unk_token or "—"
    print(f"{name:>28}  {cls:>16}  {sep:>16}  {pad:>10}  {unk:>10}")

# 모든 special token을 한 번에 보고 싶으면:
print()
for name, tok in tokenizer_specs.items():
    print(f"{name}.special_tokens_map = {tok.special_tokens_map}")

**관찰 포인트**

- **`##` 접두사 (WordPiece)**: DistilBERT·BERT는 단어 중간 서브워드를 `##xxx` 로 표시. 예: `tokenization → ['token', '##ization']`. 이전 토큰의 *연속* 이라는 신호.
- **`Ġ` 접두사 (BPE)**: GPT-2는 토큰 앞에 공백이 있었는지를 `Ġ` (Latin small letter G with stroke) 로 표시. 예: `Ġlove` 는 "love 앞에 공백이 있었다". 토큰화/디코딩이 정확히 역연산이 되도록 하는 표기.
- **대소문자**: `bert-base-cased` 는 `Hugging Face` 의 `H`, `F` 를 그대로 보존. `distilbert-base-uncased` 는 모두 소문자. 이름·고유명사 처리에서 차이가 큽니다.
- **vocab 크기**: GPT-2가 50K 로 가장 큼. BPE는 영어 외 다양한 토큰(드문 조합·바이트 단위)도 어휘에 포함하기 때문. WordPiece는 영어 중심이라 30K로 충분.
- **특수 토큰**: BERT 계열은 `[CLS]`, `[SEP]`, `[PAD]`, `[UNK]` 가 모두 정의되지만 GPT-2는 `[CLS]/[SEP]` 가 없습니다 (autoregressive 모델은 문장 시작/끝 마커를 따로 안 둠 — `<|endoftext|>` 하나가 BOS/EOS 역할). PAD도 없어 추가 설정이 필요한 경우가 흔함.

**왜 같은 문장이 다른 토큰 시퀀스가 되나?** 어휘 사전이 *사전학습 데이터* 에 따라 만들어집니다.

- BERT는 BookCorpus + Wikipedia로 학습됐고, 영어 중심 어휘.
- GPT-2는 더 다양한 웹 텍스트(Reddit 등)로 학습됐고 BPE라 어휘가 더 풍부.
- 한국어 BERT(`klue/bert-base`, Ch 14)는 한국어 코퍼스로 다시 학습돼 한국어 어휘를 보유 — 같은 문장 `"안녕"` 도 영어 BERT면 `[UNK]` 또는 글자 단위로 쪼개지지만 한국어 BERT엔 한 토큰으로 들어갑니다.

**실무 함의**: 모델을 갈아 끼울 때 토크나이저도 *반드시 짝* 으로 바꿔야 합니다. `AutoTokenizer.from_pretrained(model_name)` 의 model_name 이 모델 자체와 일치해야 하는 이유 — 학습 때 본 어휘와 추론 때 입력 어휘가 같아야 모델이 의미를 이해합니다.

## 🧩 보너스: `model.config` 안에 뭐가 있나

위에서 `model.config.id2label` 로 라벨 이름을 알아냈습니다. `config` 객체에는 모델의 *설계도* 가 모두 들어있어서, 모델을 받아왔을 때 가장 먼저 들여다보면 좋은 곳입니다.

분류 작업에서 자주 쓰는 속성들을 한 번에 출력합니다.

In [ ]:
cfg = model.config
n_params, size_mb = model_size_summary(model)   # 앞에서 정의한 헬퍼 재사용

print(f"Model name/path:          {cfg._name_or_path}")
print(f"Model type:               {cfg.model_type}")
print(f"Parameters:               {n_params:,}  ({n_params/1e6:.1f} M)")
print(f"fp32 size:                {size_mb:.1f} MB  (= params x 4 bytes)")
print(f"hidden_size:             {cfg.hidden_size}        (BERT-base/DistilBERT: 768)")
print(f"vocab_size:              {cfg.vocab_size:,}     (matches tokenizer vocab)")
print(f"max_position_embeddings: {cfg.max_position_embeddings}  (input length cap)")
print(f"num_labels:              {cfg.num_labels}          (classification head dim)")
print(f"id2label:                {cfg.id2label}")
print(f"label2id:                {cfg.label2id}")
print(f"problem_type:            {cfg.problem_type!r}    (None → auto-inferred from num_labels)")

> 📒 **더 깊이 보고 싶다면 — 부록 노트북**
>
> [`appendix_model_config.ipynb`](./appendix_model_config.ipynb) 에서 다음을 다룹니다:
> - `PretrainedConfig` 의 정체와 클래스 계층 (BertConfig / GPT2Config / T5Config / ViTConfig …)
> - `AutoConfig.from_pretrained` 로 *가중치 없이* config만 로드
> - 5종 모델(bert / distilbert / gpt2 / t5 / roberta) config를 한 표에 비교 + ViT(비전) 사례
> - 분류 헤드 갈아끼우는 `from_pretrained` 인자 패턴 (`num_labels`, `problem_type`)
> - 공식 문서 링크: <https://huggingface.co/docs/transformers/en/main_classes/configuration>
>
> Colab으로 바로: [Open](https://colab.research.google.com/github/yoon-gu/neuqes-101/blob/master/07_bert_pipeline/appendix_model_config.ipynb). 본 챕터 흐름과 별개라 시간 될 때 보시면 됩니다.

**자주 쓰는 속성 한눈에 보기**

| 속성·호출 | 의미 | 자주 쓰는 곳 |
|---|---|---|
| `model.config._name_or_path` | 모델 식별자 (Hugging Face Hub repo 또는 로컬 경로) | 어떤 모델인지 빠르게 확인 |
| `model.config.model_type` | 모델 아키텍처 종류 (`bert`, `distilbert`, `gpt2`, ...) | 분기 처리 |
| `sum(p.numel() for p in model.parameters())` | **파라미터 총 개수** (config 속성은 아니지만 항상 같이 봄) | VRAM 사용량 추정, 모델 비교 |
| `model.config.hidden_size` | hidden state 차원 (예: 768 / 1024) | 분류 헤드를 직접 만들 때 |
| `model.config.vocab_size` | 어휘 크기 (토크나이저와 일치해야 함) | 토크나이저 호환 검증 |
| `model.config.max_position_embeddings` | 입력 토큰 수 상한 | `truncation=True, max_length=...` 결정 |
| `model.config.num_labels` | 분류 헤드 출력 클래스 수 | 모델 로드 시 명시: `num_labels=5` |
| `model.config.id2label` / `label2id` | 클래스 인덱스 ↔ 이름 매핑 | 추론 결과 해석, 학습 후 모델 카드 친절도 |
| `model.config.problem_type` | `"regression"` / `"single_label_classification"` / `"multi_label_classification"` — `Trainer` 가 자동 loss 결정 | Ch 9·11·12에서 명시적으로 사용 |

**실무 패턴**: 새 모델을 받자마자 `print(model.config)` 또는 `cfg.to_dict()` 로 내용을 먼저 본다 → 입력/출력 가정을 확인하고 토크나이저·`Trainer` 설정과 일치시킴.

```python
# 새 모델 받자마자 한 줄 검사
print(model.config)            # 모든 설정 한꺼번에
print(model.config.to_dict())  # dict 형태 (JSON 직렬화 가능)
```

## 📦 이번 챕터에 등장한 라이브러리·함수

### `transformers`

| 이름 | 한 줄 설명 | 다음 챕터에서 |
|---|---|---|
| `transformers.pipeline` | 추론 원스톱 함수 (3단계 묶음) | 학습된 모델을 `pipeline`으로 감싸 사용 (Ch 9 이후) |
| `transformers.AutoTokenizer` | 모델에 맞는 토크나이저 자동 로드 | Ch 8에서 옵션 깊게 보기 |
| `transformers.AutoModelForSequenceClassification` | 분류 헤드가 붙은 모델 자동 로드 | Ch 9부터 직접 파인튜닝 |

### `torch` 의 후처리·연산 함수

| 형태 | 호출 예시 | 언제 |
|---|---|---|
| `torch.softmax(x, dim=-1)` | 텐서에 직접 | 추론 후처리 (이번 챕터처럼) |
| `torch.nn.functional.softmax` (보통 `F.softmax`) | `F.softmax(x, dim=-1)` | 함수형, PyTorch 코드에 가장 흔함 |
| `torch.nn.Softmax(dim=-1)` | layer로 모델 안에 박을 때 | 잘 안 씀 (보통 logits 그대로 두고 loss가 처리) |
| `F.log_softmax`, `torch.log_softmax` | softmax + log 한 번에 (수치 안정) | 학습 loss 직접 구현 시 |
| `torch.argmax(x, dim=-1)` 또는 `x.argmax(dim=-1)` | 가장 큰 값의 인덱스 | 분류 예측 인덱스 추출 |
| `torch.no_grad()` (context manager) | 추론 중 gradient 비활성 | 메모리·속도 절약 |

> **요점**: HuggingFace는 softmax·argmax 같은 *수치 함수* 를 따로 제공하지 않습니다. 모두 PyTorch에서 직접 호출합니다 (TensorFlow 백엔드를 쓰면 `tf.nn.softmax` 같은 식). `Trainer` 가 학습 loss 안에서 softmax를 자동 처리하므로, 학습 코드에서는 직접 부를 일이 거의 없고 *추론 후처리* 에서 등장하는 게 보통.

### `model.config` 의 자주 쓰는 속성

| 속성 | 용도 |
|---|---|
| `id2label`, `label2id` | 클래스 인덱스 ↔ 이름 매핑 |
| `num_labels` | 분류 헤드 출력 차원 |
| `hidden_size`, `vocab_size`, `max_position_embeddings` | 모델 구조 파라미터 |
| `model_type`, `_name_or_path` | 모델 정체성 |
| `problem_type` | `Trainer` 자동 loss 선택 (`regression` / `single_label_classification` / `multi_label_classification`) |

### `torch` 자체

| 이름 | 한 줄 설명 | 다음 챕터에서 |
|---|---|---|
| `torch` | PyTorch — 텐서 연산, 자동 미분, GPU 연결 | 계속 사용 (특히 Ch 9 학습부터) |

## 🎯 체크포인트 질문

1. `pipeline("sentiment-analysis")` 을 직접 풀어 쓰면 어떤 단계가 되는지 4단계로 설명할 수 있나요?
2. `input_ids` 와 `attention_mask` 는 각각 무슨 역할인가요?
3. 모델 출력의 `logits` 는 왜 그대로 쓰지 않고 `softmax` 를 거치나요? (Ch 4에서 본 동등성 떠올리기)
4. `AutoTokenizer.from_pretrained(...)` 를 쓸 때와 `BertTokenizer.from_pretrained(...)` 를 직접 쓸 때의 차이는 무엇인가요?

## ❓ FAQ

### Q1. (실무) `pipeline` 이 처음 실행될 때 너무 느린데 정상인가요?

네, 정상입니다. 첫 실행 시 다음이 한꺼번에 일어납니다.

1. 모델 가중치 다운로드 (약 250MB for DistilBERT)
2. 토크나이저 사전 다운로드
3. 모델 PyTorch 로드 + (있다면) GPU 이동

두 번째부터는 캐시(`~/.cache/huggingface/`)에서 읽으므로 즉시 실행됩니다. **Colab 세션이 끊기면 캐시도 사라져** 다시 다운로드합니다 — 자주 쓴다면 Google Drive를 마운트해서 `HF_HOME` 환경변수를 Drive로 지정하면 보존됩니다.

```python
# Drive에 캐시 보존 (선택)
import os
from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
```

### Q2. (이론) 왜 BERT 토크나이저는 단어를 `##` 조각으로 쪼개나요?

이게 **WordPiece** 의 핵심입니다 — 자주 등장하는 부분 문자열을 하나의 토큰으로 두고, 새로운 단어는 *서브워드* 들의 조합으로 표현합니다.

```python
# 사전학습 어휘에 없는 단어를 어떻게 처리하는지 보기
tokenizer.tokenize("Tokenization")     # ['token', '##ization']
tokenizer.tokenize("antidisestablishmentarianism")
# → ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']
```

`##` 은 "이 토큰은 *이전 토큰의 연속*"이라는 표시입니다.

**왜 이렇게?**
- **OOV 해결**: TF-IDF는 학습 어휘에 없는 단어를 *무시* 했지만, WordPiece는 항상 더 작은 서브워드로 쪼개 표현 가능 (이론적으로 OOV 없음, 최악의 경우 글자 단위까지).
- **어휘 크기 관리**: 영어에는 수백만 단어가 있지만 BERT는 30,522개 토큰만으로 모두 표현. 형태 변형(`-ing`, `-ed`)도 일관되게 처리.
- **희귀 단어 일반화**: `unhappiness` 를 `un + happi + ness` 로 보면, 모델이 `happi` 를 알고 있으면 처음 보는 단어라도 의미 추론 가능.

### Q3. (실무) `pipeline` 결과의 `LABEL_0`, `LABEL_1` 이 무슨 의미인지 어떻게 알아내나요?

`model.config.id2label` 을 확인하면 됩니다.

```python
print(classifier.model.config.id2label)
# {0: 'NEGATIVE', 1: 'POSITIVE'} — 이 모델은 친절히 적어둠
```

**모델마다 다릅니다**. 일부 모델은 `LABEL_0`, `LABEL_1` 처럼 *추상적인* 이름만 붙어 있어 모델 카드(Hugging Face Hub의 모델 페이지) 또는 학습 데이터셋 라벨 정의를 확인해야 합니다. 우리가 Ch 9 이후 직접 학습할 때는 `id2label` 을 명시적으로 설정해 미래의 사용자에게 친절하게 만들 수 있습니다.

### Q4. (이론) `[CLS]` 와 `[SEP]` 토큰은 왜 필요한가요?

BERT의 사전학습 구조에서 비롯된 특수 토큰입니다.

- **`[CLS]`** (Classification): 입력 맨 앞에 항상 붙는 토큰. 사전학습 시 *Next Sentence Prediction* 을 위한 자리였고, 분류 작업에서는 이 위치의 hidden state(전체 문장의 표현)를 분류 헤드에 넣습니다. *모든 토큰의 정보가 attention을 통해 [CLS]로 모이도록* 학습됨.
- **`[SEP]`** (Separator): 문장 끝에 붙거나, 두 문장을 분리. 입력이 한 문장이면 `[CLS] ... [SEP]` 구조, 두 문장(질문-답변 등)이면 `[CLS] ... [SEP] ... [SEP]`.

이번 챕터의 입력 `"I love using Hugging Face!"` 의 token ID 첫 값 `101` 이 `[CLS]`, 마지막 `102` 가 `[SEP]` — 위 셀에서 확인했죠.

`AutoTokenizer` 가 이 특수 토큰을 자동으로 추가해주므로 우리가 신경 쓸 일은 거의 없지만, *왜 길이가 입력 단어 수보다 2 길게 나오는지* 이해하는 데는 이 두 토큰을 알아야 합니다.

### Q5. (실무) GPU 없이도 `pipeline` 이 돌아가나요?

네, CPU에서도 작동합니다. 다만 속도 차이가 큽니다.

| 환경 | DistilBERT 1문장 추론 |
|---|---|
| Colab CPU | ~80-150ms |
| Colab T4 GPU | ~5-15ms |
| 큰 BERT 모델 | CPU는 5-10x 더 느림 |

**추론** 만 한다면 CPU도 실용적입니다 (예: API 서빙은 보통 GPU지만 가벼운 데모는 CPU). **학습** 은 거의 항상 GPU 필수 — Ch 9 이후 학습할 때는 T4 런타임으로 바꿔야 합니다.

### Q6. (실무) `AutoTokenizer` 와 `BertTokenizer` 를 직접 사용하는 것의 차이는?

기능적으론 거의 같지만 **코드 일반성** 이 다릅니다.

```python
# 방식 A: 직접 클래스 (모델 종류를 코드에 박음)
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# 만약 GPT-2를 쓰려면 → import 와 클래스 모두 바꿔야 함

# 방식 B: Auto (모델 이름만 주면 알아서)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# GPT-2로 바꾸려면 → 문자열만 "gpt2"로 변경
```

실무에선 **거의 항상 `AutoTokenizer`** 를 씁니다. `Auto` 계열은 모델 카드(`config.json`) 에서 어떤 클래스를 써야 할지 자동 추론하므로, 다른 모델로 갈아끼우는 실험이 매우 쉽습니다. Ch 14의 한국어 BERT(`klue/bert-base`) 도 같은 패턴으로 로드됩니다 — 코드 한 줄도 안 바꾸고요.

## 🚀 삽질 코너 (선택)

다음 코드를 돌려보고 에러 메시지를 읽어보세요. 어떤 인자가 빠졌을까요?

```python
# 에러가 나는 코드
inputs_bad = tokenizer("I love HF!")        # return_tensors 빠짐
outputs_bad = model(**inputs_bad)
```

힌트: 모델은 PyTorch 텐서를 기대하는데, 토크나이저가 기본값으로 무엇을 반환할까요? `tokenizer("...")` 의 기본 반환 형식과 `tokenizer("...", return_tensors="pt")` 의 차이를 출력 비교해보세요.

## 다음 챕터 예고

**Chapter 8. Tokenizer 깊게 보기 + Datasets 라이브러리**

- 서브워드 토큰화 옵션 — `padding`, `truncation`, `max_length` 의 의미
- `datasets` 라이브러리: `load_dataset`, `map`, `filter`, Apache Arrow 메모리 효율
- DataLoader 변환 — Ch 9 학습 코드의 입력 준비 단계
- **여전히 학습 없음** (추론 데이터 파이프라인까지만)